# E5-base fine-tune + GPU endpoint cho Research-assist

Chọn **intfloat/multilingual-e5-base** cho lượt đầu: encoder đa ngôn ngữ, prefix
`query:` / `passage:` nhất quán, pipeline CachedMNRL đã có sẵn trong project.
Đây là lựa chọn thực dụng về tích hợp/huấn luyện, không phải tuyên bố model tốt
nhất mọi dataset. Qwen3-Embedding-0.6B là đối chứng sau khi có baseline riêng.

Notebook train dùng một GPU T4, không cộng VRAM của hai T4. QASPER mặc định là
tiếng Anh; bổ sung dữ liệu Việt–Anh đã review để đo và cải thiện đúng miền sử dụng.
Các model LLM planning/writing vẫn dùng cấu hình riêng của app.

Các cell cuối **tùy chọn** chạy API GPU sau khi train. Cloudflare chỉ chuyển tiếp
HTTP; GPU đến từ máy đang chạy notebook. URL tạm hết hiệu lực khi tunnel dừng.
Không coi phiên notebook là dịch vụ 24/7. Với Kaggle, kiểm tra điều khoản hiện hành
và chỉ dùng endpoint nếu hình thức sử dụng này được nền tảng cho phép; không có
keep-alive hay cơ chế vượt quota. Có thể dùng phần serve trên GPU host cho phép.

Nếu train gần hết giờ phiên, tải checkpoint ZIP, mở notebook 04 trong phiên mới,
thêm ZIP làm Input rồi chỉ chạy inference. Không train và serve cùng lúc.

Nguồn chính thức:
- https://huggingface.co/intfloat/multilingual-e5-base
- https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
- https://sbert.net/docs/package_reference/sentence_transformer/losses.html
- https://www.kaggle.com/docs/notebooks
- https://www.kaggle.com/terms
- https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/

## Phiên inference riêng
Thêm checkpoint ZIP từ notebook 01/03 làm Input. Notebook này không train lại model.

In [ ]:
"""Notebook bootstrap: verify a fresh process and stop before using stale native modules."""

import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import uuid
from pathlib import Path

TRAINING_REQUIREMENTS = [
    "sentence-transformers==5.1.2",
    "transformers==4.57.1",
    "datasets==4.3.0",
    "accelerate==1.11.0",
    "huggingface-hub==0.36.0",
    "pyarrow==21.0.0",
    "numpy==2.2.6",
    "sentencepiece==0.2.1",
]
WATCHED_MODULES = {
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "torch": "torch",
    "transformers": "transformers",
    "sentence_transformers": "sentence-transformers",
    "datasets": "datasets",
    "huggingface_hub": "huggingface-hub",
    "tokenizers": "tokenizers",
    "accelerate": "accelerate",
    "safetensors": "safetensors",
}
PREFLIGHT_CODE = r"""
import json, sys
import numpy as np
from numpy import strings
assert strings.add(["a"], ["b"]).tolist() == ["ab"]
from scipy.sparse import csr_matrix
from sklearn.metrics import pairwise_distances
import pandas, pyarrow, datasets, torch
from sentence_transformers import (
    SentenceTransformer, CrossEncoder,
    SentenceTransformerTrainer, SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
assert csr_matrix([[1.0]]).shape == (1, 1)
assert pairwise_distances([[0.0], [1.0]]).shape == (2, 2)
print(json.dumps({"imports_ok": True, "python": sys.version.split()[0],
                  "numpy": np.__version__, "numpy_file": np.__file__,
                  "torch": torch.__version__, "cuda_available": torch.cuda.is_available()}))
"""


def installed_versions():
    versions = {}
    for name in set(WATCHED_MODULES.values()) | {r.split("==")[0] for r in TRAINING_REQUIREMENTS}:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def loaded_versions():
    return {
        name: vars(sys.modules[name]).get("__version__")
        for name in WATCHED_MODULES
        if name in sys.modules and sys.modules[name] is not None
    }


def current_runtime_identity():
    # A restarted container/kernel may reuse the same PID. A token stored on sys
    # survives re-running the cell, but cannot survive a real Python restart.
    if not getattr(sys, "_researchdesk_runtime_id", None):
        sys._researchdesk_runtime_id = f"{os.getpid()}:{uuid.uuid4().hex}"
    return sys._researchdesk_runtime_id


def install_training_environment(marker_path, runtime_id=None):
    """Never treat a successful pip install as proof the current kernel is coherent."""
    runtime_id = runtime_id or current_runtime_identity()
    marker_path = Path(marker_path)
    before, loaded = installed_versions(), loaded_versions()
    needed = [r for r in TRAINING_REQUIREMENTS if before[r.split("==")[0]] != r.split("==")[1]]
    if needed:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--prefer-binary",
                "--only-binary=numpy,pyarrow",
                "--upgrade-strategy",
                "only-if-needed",
                *TRAINING_REQUIREMENTS,
            ],
            check=True,
        )
    # A child process has no old NumPy C-extension objects from the notebook kernel.
    checked = subprocess.run(
        [sys.executable, "-c", PREFLIGHT_CODE],
        capture_output=True,
        text=True,
        timeout=180,
    )
    repaired = False
    diagnostics = (checked.stdout or "") + (checked.stderr or "")
    if checked.returncode and any(
        text in diagnostics
        for text in (
            "numpy.ufunc",
            "numpy.dtype size changed",
            "numpy._core",
            "numpy.core",
        )
    ):
        # Repair only NumPy's files; do not reinstall Kaggle's CUDA PyTorch.
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--force-reinstall",
                "--no-cache-dir",
                "--no-deps",
                "--only-binary=:all:",
                "numpy==2.2.6",
            ],
            check=True,
        )
        repaired = True
        checked = subprocess.run(
            [sys.executable, "-c", PREFLIGHT_CODE],
            capture_output=True,
            text=True,
            timeout=180,
        )
    if checked.returncode:
        print((checked.stdout or "")[-4000:])
        print((checked.stderr or "")[-12000:])
        raise RuntimeError(
            "Fresh-process dependency check failed. Training has NOT started. "
            "Keep the complete output above for diagnosis."
        )
    after = installed_versions()
    fingerprint = hashlib.sha256(json.dumps(after, sort_keys=True).encode()).hexdigest()
    previous = json.loads(marker_path.read_text(encoding="utf-8")) if marker_path.exists() else {}
    changed = before != after
    mismatch = any(
        version is not None and str(version) != str(after.get(WATCHED_MODULES[name]))
        for name, version in loaded.items()
    )
    pending_same_kernel = previous.get("restart_required_for") == runtime_id
    unknown_loaded_kernel = bool(loaded) and (
        previous.get("fingerprint") != fingerprint
        or previous.get("runtime_identity_format") != "memory-token-v1"
    )
    restart = (
        pending_same_kernel
        or bool(loaded)
        and (changed or repaired or mismatch or unknown_loaded_kernel)
    )
    status = {
        "ready": not restart,
        "fingerprint": fingerprint,
        "versions": after,
        "runtime_id": runtime_id,
        "runtime_identity_format": "memory-token-v1",
        "restart_required_for": runtime_id if restart else None,
        "loaded_modules_before_install": sorted(loaded),
        "fresh_process_output": checked.stdout.strip(),
        "numpy_repaired": repaired,
    }
    marker_path.parent.mkdir(parents=True, exist_ok=True)
    marker_path.write_text(json.dumps(status, indent=2), encoding="utf-8")
    print(checked.stdout.strip())
    if restart:
        print(
            "THIẾT LẬP ĐÃ THÀNH CÔNG — cần RESTART KERNEL/PYTHON một lần.\n"
            "Đây là bước hoàn tất cài đặt, không phải lỗi huấn luyện.\n"
            "Sau restart, chạy lại từ cell đầu. Các cell phía sau sẽ tạm bỏ qua "
            "để tránh dùng thư viện cũ trong RAM."
        )
        return status
    print("Environment ready: fresh-process imports passed and no kernel restart is pending.")
    return status


import os, sys, subprocess
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
_RESEARCHDESK_ENV_READY = False
status = install_training_environment("/kaggle/temp/researchdesk-inference-environment.json")
_RESEARCHDESK_ENV_READY = status["ready"]
if _RESEARCHDESK_ENV_READY:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "fastapi>=0.115,<1", "uvicorn>=0.30,<1", "httpx>=0.27,<1"])
    print("Environment ready. Run the checkpoint selection cell.")

## GPU inference và Cloudflare Tunnel (tùy chọn)
Giữ notebook ở chế độ tương tác; không commit/save output có secret. Checkpoint và model ID được giữ riêng với URL tạm. Kaggle không phải hosting luôn bật; kiểm tra quyền sử dụng nền tảng trước khi bật tunnel.

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.115,<1", "uvicorn>=0.30,<1", "httpx>=0.27,<1"])

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False):
    """Authenticated, bounded E5 inference; usable on a permitted notebook or GPU host."""

    import asyncio
    import hashlib
    import json
    import secrets
    import threading
    from pathlib import Path
    from typing import Literal

    from fastapi import FastAPI, HTTPException, Request
    from pydantic import BaseModel, ConfigDict, ValidationError


    def checkpoint_identity(path, max_seq_length=512):
        """Identity follows weights/tokenizer/config, never the temporary tunnel URL."""
        root = Path(path)
        files = sorted(
            p
            for p in root.rglob("*")
            if p.is_file()
            and p.suffix
            in {
                ".safetensors",
                ".bin",
                ".json",
                ".model",
                ".txt",
            }
            and "checkpoints" not in p.relative_to(root).parts
        )
        if not any(p.suffix in {".safetensors", ".bin"} for p in files):
            raise ValueError("Checkpoint must contain model weights and tokenizer/config files.")
        digest = hashlib.sha256(f"e5:query:passage:normalized:v1:{max_seq_length}".encode())
        for file in files:
            digest.update(str(file.relative_to(root)).replace("\\", "/").encode())
            with file.open("rb") as stream:
                while block := stream.read(1024 * 1024):
                    digest.update(block)
        return "research-e5-" + digest.hexdigest()[:32]


    class EmbeddingInput(BaseModel):
        model_config = ConfigDict(extra="forbid", strict=True)
        model: str
        input: str | list[str]
        input_type: Literal["query", "passage"]


    def create_embedding_app(encoder, api_key, model_id, max_seq_length=512):
        if len(api_key) < 32:
            raise ValueError("Use an API key with at least 32 characters.")
        app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
        lock = threading.Lock()
        active = 0
        dimension = int(encoder.get_sentence_embedding_dimension())

        def authorize(request):
            header = request.headers.get("authorization", "")
            if not secrets.compare_digest(header, "Bearer " + api_key):
                raise HTTPException(401, "Invalid API key.")

        @app.get("/health")
        async def health():
            return {"status": "ok"}

        @app.get("/v1/info")
        async def info(request: Request):
            authorize(request)
            return {
                "model": model_id,
                "family": "e5",
                "dimension": dimension,
                "max_seq_length": max_seq_length,
                "normalized": True,
                "protocol": "research-embedding-v1",
                "max_batch_size": 32,
            }

        def encode(texts, input_type):
            with lock:
                prefix = "query: " if input_type == "query" else "passage: "
                vectors = encoder.encode(
                    [prefix + t for t in texts],
                    batch_size=8,
                    normalize_embeddings=True,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                )
                return vectors.tolist()

        @app.post("/v1/embeddings")
        async def embeddings(request: Request):
            nonlocal active
            authorize(request)
            if active >= 4:
                raise HTTPException(429, "GPU queue is full; retry later.")
            body = bytearray()
            async for part in request.stream():
                body.extend(part)
                if len(body) > 300000:
                    raise HTTPException(413, "Request is too large.")
            try:
                value = EmbeddingInput.model_validate_json(body)
            except (ValidationError, ValueError):
                raise HTTPException(422, "Invalid embedding request.") from None
            if value.model != model_id:
                raise HTTPException(409, "Checkpoint changed; configure the new model ID and reindex.")
            texts = [value.input] if isinstance(value.input, str) else value.input
            if not 1 <= len(texts) <= 32 or any(
                not text.strip() or len(text) > 16000 for text in texts
            ):
                raise HTTPException(422, "Use 1-32 nonempty texts, at most 16000 characters each.")
            if active >= 4:
                raise HTTPException(429, "GPU queue is full; retry later.")
            # Retain the queue slot and physical lock until the thread actually exits.
            active += 1
            job = asyncio.create_task(asyncio.to_thread(encode, texts, value.input_type))

            def finished(task):
                nonlocal active
                active -= 1
                if not task.cancelled():
                    task.exception()  # Retrieve a late exception after an HTTP cancellation.

            job.add_done_callback(finished)
            try:
                vectors = await asyncio.shield(job)
            except asyncio.CancelledError:
                raise
            except Exception:
                raise HTTPException(
                    503, "GPU inference failed; inspect the private server log."
                ) from None
            return {
                "object": "list",
                "model": model_id,
                "data": [
                    {"object": "embedding", "index": i, "embedding": vector}
                    for i, vector in enumerate(vectors)
                ],
            }

        return app


    def extract_checkpoint(archive, output):
        """Extract a downloaded training checkpoint; reject traversal and symlinks."""
        import zipfile

        root = Path(output).resolve()
        if root.exists() and any(root.iterdir()):
            raise ValueError("Choose an empty checkpoint extraction directory.")
        with zipfile.ZipFile(archive) as zipped:
            total = 0
            for item in zipped.infolist():
                target = (root / item.filename.replace("\\", "/")).resolve()
                if not target.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe checkpoint archive entry.")
                total += item.file_size
                if total > 5 * 1024**3:
                    raise ValueError("Checkpoint archive exceeds 5 GiB.")
            zipped.extractall(root)
        candidates = list(root.rglob("modules.json"))
        if len(candidates) != 1:
            raise ValueError("Use the checkpoint ZIP, not the full experiment/checkpoints ZIP.")
        return candidates[0].parent


    def load_e5_checkpoint(path, device="cuda", max_seq_length=512):
        from sentence_transformers import SentenceTransformer

        root = Path(path)
        manifest = json.loads((root / "research_model.json").read_text(encoding="utf-8"))
        if manifest.get("model_family") != "e5" or not manifest.get("fine_tuned"):
            raise ValueError("Select an E5 checkpoint with actual optimizer steps (fine_tuned=true).")
        model = SentenceTransformer(
            str(root), device=device, local_files_only=True, trust_remote_code=False
        )
        model.max_seq_length = max_seq_length
        return model, checkpoint_identity(root, max_seq_length)

In [ ]:
# Train+serve: MODEL_OUT is set by training above. In a fresh notebook, use ZIP or path.
CHECKPOINT_PATH = str(globals().get("MODEL_OUT", ""))
CHECKPOINT_ZIP = ""  # e.g. /kaggle/input/my-e5/research-e5-checkpoint.zip
PORT = 8008
MAX_SEQ_LENGTH = 512
START_GPU_SERVER = False  # Run All can train/export without waiting for an API key
START_CLOUDFLARE = False  # opt in only on a host/platform permitting remote inference
API_KEY_SECRET_NAME = "RESEARCH_GPU_API_KEY"  # create this Kaggle Secret, >=32 chars

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False) and START_GPU_SERVER:
    import gc, getpass, os, threading, time
    import torch, uvicorn, httpx
    from IPython.display import display, Markdown
    if "GPU_SERVER" in globals():
        raise RuntimeError("Stop the previous server using the cleanup cell before starting again.")
    # If training ran in this kernel, release its model/optimizer before serving.
    gc.collect()
    torch.cuda.empty_cache()
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator before starting this endpoint.")
    if CHECKPOINT_ZIP:
        selected_path = extract_checkpoint(CHECKPOINT_ZIP, "/kaggle/temp/research-e5-serve")
    elif CHECKPOINT_PATH and Path(CHECKPOINT_PATH).is_dir():
        selected_path = Path(CHECKPOINT_PATH)
    else:
        found = list(Path("/kaggle/input").rglob("research-e5-checkpoint.zip"))
        if len(found) != 1:
            raise RuntimeError("Set CHECKPOINT_ZIP or CHECKPOINT_PATH to the trained checkpoint.")
        selected_path = extract_checkpoint(found[0], "/kaggle/temp/research-e5-serve")
    try:
        from kaggle_secrets import UserSecretsClient
        API_KEY = UserSecretsClient().get_secret(API_KEY_SECRET_NAME)
    except Exception:
        API_KEY = getpass.getpass("Enter your private GPU API key (at least 32 characters): ")
    if len(API_KEY) < 32:
        raise ValueError("Use at least 32 characters; put the same key in your local app config.")
    GPU_ENCODER, GPU_MODEL_ID = load_e5_checkpoint(selected_path, "cuda", MAX_SEQ_LENGTH)
    GPU_APP = create_embedding_app(GPU_ENCODER, API_KEY, GPU_MODEL_ID, MAX_SEQ_LENGTH)
    GPU_SERVER = uvicorn.Server(uvicorn.Config(GPU_APP, host="127.0.0.1", port=PORT,
                               log_level="warning", access_log=False))
    GPU_THREAD = threading.Thread(target=GPU_SERVER.run, daemon=True)
    GPU_THREAD.start()
    deadline = time.monotonic() + 60
    while not GPU_SERVER.started and GPU_THREAD.is_alive() and time.monotonic() < deadline:
        time.sleep(0.2)
    if not GPU_SERVER.started:
        raise RuntimeError("Server did not start; inspect the private notebook log.")
    local_url = f"http://127.0.0.1:{PORT}"
    check = httpx.post(local_url + "/v1/embeddings", headers={"Authorization": "Bearer " + API_KEY},
        json={"model": GPU_MODEL_ID, "input": ["How is evidence retrieved?"],
              "input_type": "query"}, timeout=60)
    check.raise_for_status()
    print("GPU:", torch.cuda.get_device_name(0))
    print("Model ID:", GPU_MODEL_ID)
    print("Embedding dimension:", len(check.json()["data"][0]["embedding"]))
    print("Local API test passed. The API key is intentionally not printed/saved.")
elif globals().get("_RESEARCHDESK_ENV_READY", False):
    print("Server disabled. Set START_GPU_SERVER=True when you want to serve the checkpoint.")

In [ ]:
if globals().get("_RESEARCHDESK_ENV_READY", False) and START_CLOUDFLARE:
    import hashlib, json, re, shutil, subprocess, threading, time, urllib.request
    if "GPU_SERVER" not in globals() or not GPU_SERVER.started:
        raise RuntimeError("Start the GPU server and pass its local API check before tunneling.")
    if "CLOUDFLARE_PROCESS" in globals() and CLOUDFLARE_PROCESS.poll() is None:
        raise RuntimeError("Stop the existing tunnel before creating another one.")
    binary = shutil.which("cloudflared")
    if not binary:
        # Resolve an official release and verify GitHub's published asset digest.
        req = urllib.request.Request("https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
                                     headers={"User-Agent": "Research-assist-notebook"})
        with urllib.request.urlopen(req, timeout=30) as response:
            release = json.load(response)
        asset = next(a for a in release["assets"] if a["name"] == "cloudflared-linux-amd64")
        digest = asset.get("digest", "")
        if not digest.startswith("sha256:"):
            raise RuntimeError("No published SHA256 digest. Install verified cloudflared manually.")
        binary = "/kaggle/temp/cloudflared"
        urllib.request.urlretrieve(asset["browser_download_url"], binary)
        with open(binary, "rb") as stream:
            actual = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual != digest.split(":", 1)[1]:
            raise RuntimeError("cloudflared checksum mismatch; refusing execution.")
        os.chmod(binary, 0o700)
    CLOUDFLARE_PROCESS = subprocess.Popen([binary, "tunnel", "--no-autoupdate", "--url",
        f"http://127.0.0.1:{PORT}"], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    TUNNEL_URL = None
    TUNNEL_READY = threading.Event()
    def read_tunnel_log():
        global TUNNEL_URL
        for line in CLOUDFLARE_PROCESS.stderr:
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if match:
                TUNNEL_URL = match.group(0)
                TUNNEL_READY.set()
    threading.Thread(target=read_tunnel_log, daemon=True).start()
    if not TUNNEL_READY.wait(60):
        CLOUDFLARE_PROCESS.terminate()
        raise RuntimeError("No tunnel URL received. Check network/platform permissions.")
    print("Copy this URL:", TUNNEL_URL)
    print("\nLocal app .env (replace existing entries):")
    print("RA_EMBEDDING_PROVIDER=remote")
    print("RA_EMBEDDING_BASE_URL=" + TUNNEL_URL + "/v1")
    print("RA_EMBEDDING_MODEL=" + GPU_MODEL_ID)
    print("RA_EMBEDDING_API_KEY=<the same private key you entered above>")
    print("RA_EMBEDDING_MAX_SEQ_LENGTH=" + str(MAX_SEQ_LENGTH))
    print("RA_EMBEDDING_BATCH_SIZE=16")
    print("RA_CHUNK_STRATEGY=tokens")
    print("RA_CHUNK_TOKENS=384")
    print("RA_CHUNK_TOKEN_OVERLAP=48")
    print("RA_CHUNK_TOKENIZER_MODEL=intfloat/multilingual-e5-base")
    print("Restart the app. A new checkpoint requires Reindex.")
    print("A new URL for the same model ID does not require Reindex.")
elif globals().get("_RESEARCHDESK_ENV_READY", False):
    print("Tunnel disabled. Set START_CLOUDFLARE=True only if your host permits it.")

In [ ]:
# Run explicitly when you finish. This cell is not part of Run All.
STOP_ENDPOINT = False
if STOP_ENDPOINT:
    if "CLOUDFLARE_PROCESS" in globals() and CLOUDFLARE_PROCESS.poll() is None:
        CLOUDFLARE_PROCESS.terminate()
        CLOUDFLARE_PROCESS.wait(timeout=15)
    if "GPU_SERVER" in globals():
        GPU_SERVER.should_exit = True
        GPU_THREAD.join(timeout=60)
        if GPU_THREAD.is_alive():
            raise RuntimeError("GPU work is still draining. Wait before unloading the model.")
        del GPU_SERVER, GPU_APP, GPU_ENCODER
        gc.collect()
        torch.cuda.empty_cache()
    print("Endpoint stopped.")